Priprema podataka za obradu

In [1]:
!pip install scikit-surprise --quiet 2>/dev/null
!pip install "numpy==1.26.4" --force-reinstall --quiet 2>/dev/null
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import os
ml_1m_url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
urllib.request.urlretrieve(ml_1m_url, "ml-1m.zip")
with zipfile.ZipFile("ml-1m.zip", "r") as ml_1m_zip:
  ml_1m_zip.extractall()
movies = pd.read_csv("ml-1m/movies.dat", sep="::", names=["movie_id", "title", "genres"], engine="python", encoding="latin-1")
users = pd.read_csv("ml-1m/users.dat", sep="::", names=["user_id", "gender", "age", "occupation", "zipcode"], engine="python", encoding="latin-1")
ratings = pd.read_csv("ml-1m/ratings.dat", sep="::", names=["user_id", "movie_id", "rating", "timestamp"], engine="python", encoding="latin-1")
genres = {"Action": 0, "Adventure": 1, "Animation": 2, "Children's": 3, "Comedy": 4, "Crime": 5, "Documentary": 6, "Drama": 7, "Fantasy": 8, "Film-Noir": 9, "Horror": 10, "Musical": 11, "Mystery": 12, "Romance": 13, "Sci-Fi": 14, "Thriller": 15, "War": 16, "Western": 17}
ages = {1: 0 , 18:  1, 25:  2, 35:  3 , 45:  4 , 50:  5, 56:  6};
occupations = {0:  "other or not specified",  1:  "academic/educator",  2:  "artist",  3:  "clerical/admin",  4:  "college/grad student",  5:  "customer service",  6:  "doctor/health care",  7:  "executive/managerial",  8:  "farmer",  9:  "homemaker", 10:  "K-12 student", 11:  "lawyer" , 12:  "programmer", 13:  "retired", 14:  "sales/marketing", 15:  "scientist", 16:  "self-employed", 17:  "technician/engineer", 18:  "tradesman/craftsman" , 19:  "unemployed", 20:  "writer"}

Priprema tenzora, klasa Model i priprema za učenje

In [2]:
def create_movies_tensor(movies, genres):
  movies_values = movies.values
  for i in range(len(movies_values)):
    movies_values[i, 2] = movies_values[i, 2].split("|")
    ind_genres = []
    for j in range(len(genres)):
      ind_genres.append(0)
    for j in range(len(movies_values[i, 2])):
      ind_genres[genres[movies_values[i,2][j]]] = 1
    movies_values[i, 2] = ind_genres
  movies_values = np.delete(movies_values, 1, axis=1)
  movie_ids = movies_values[:, 0].astype(np.float32)
  movie_ids_tensor = torch.tensor(movie_ids).unsqueeze(1)
  movie_genres = np.array(list(movies_values[:, 1]), dtype=np.float32)
  movie_genres_tensor = torch.tensor(movie_genres)
  movies_tensor = torch.cat((movie_ids_tensor, movie_genres_tensor), 1)
  return movies_tensor
def create_users_tensor(users):
  user_values = users.values
  for i in range(len(user_values)):
    if user_values[i, 1] == "M":
      user_values[i, 1] = 0
    else:
      user_values[i, 1] = 1
    user_values[i, 2] = ages[user_values[i, 2]]
  user_values = np.delete(user_values, 4, axis=1)
  user_list = user_values.astype(np.float32)
  user_tensor = torch.tensor(user_list)
  return user_tensor
def create_ratings_tensor(ratings):
  rating_values = ratings.values
  rating_values = np.delete(rating_values, 3, axis=1)
  rating_list = rating_values.astype(np.float32)
  rating_tensor = torch.tensor(rating_list)
  return rating_tensor

movies_tensor = create_movies_tensor(movies, genres)
user_tensor = create_users_tensor(users)
rating_tensor = create_ratings_tensor(ratings)

class Model(nn.Module):
  def __init__(self, movies, users):
    super().__init__()
    self.max_movies_id = int(movies.values[:, 0].max())
    self.max_users_id = int(users.values[:, 0].max())
    self.movies_embedding = nn.Embedding(self.max_movies_id + 1, 64)
    self.users_embedding = nn.Embedding(self.max_users_id + 1, 64)
    self.linear1 = nn.Linear(149, 128)
    self.linear2 = nn.Linear(128, 64)
    self.linear3 = nn.Linear(64, 1)
    self.batchnorm1 = nn.BatchNorm1d(128)
    self.batchnorm2 = nn.BatchNorm1d(64)
    self.dropout = nn.Dropout(0.2)
  def forward(self, user_id, movie_id, user_details, movie_details):
    if user_id is not None:
      users_embedding = self.users_embedding(user_id.long())
    else:
      users_embedding = torch.zeros(1, 64)
    vector = torch.cat([self.movies_embedding(movie_id.long()), users_embedding, user_details, movie_details], dim=1)
    vector = self.dropout(torch.relu(self.batchnorm1(self.linear1(vector))))
    vector = self.dropout(torch.relu(self.batchnorm2(self.linear2(vector))))
    vector = self.linear3(vector)
    return vector

model = Model(movies, users)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

user_dict = {}
for i in range(len(user_tensor)):
    user_id = int(user_tensor[i, 0].item())
    user_dict[user_id] = i
movie_dict = {}
for i in range(len(movies_tensor)):
    movie_id = int(movies_tensor[i, 0].item())
    movie_dict[movie_id] = i

user_id_per_rating = []
movie_id_per_rating = []
for i in range(len(rating_tensor)):
    user_id  = int(rating_tensor[i, 0])
    movie_id = int(rating_tensor[i, 1])
    user_id_per_rating.append(user_dict[user_id])
    movie_id_per_rating.append(movie_dict[movie_id])
user_id_per_rating  = torch.tensor(user_id_per_rating,  dtype=torch.long)
movie_id_per_rating = torch.tensor(movie_id_per_rating, dtype=torch.long)

train_data_len = int(0.8*len(rating_tensor))
perm = torch.randperm(len(rating_tensor))
train_data = perm[:train_data_len]
test_data = perm[train_data_len:]

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)


Učenje modela (ispod se nalazi opcija za učitavanje već istreniranog modela)

In [6]:
from torch.utils.data import TensorDataset, DataLoader

train_tensordataset = TensorDataset(rating_tensor[train_data, 0], rating_tensor[train_data, 1], user_id_per_rating[train_data], movie_id_per_rating[train_data], rating_tensor[train_data, 2])
train_dataloader = DataLoader(train_tensordataset, batch_size=2048, shuffle=True)
epochs = 40
for epoch in range(epochs):
  model.train()
  total_loss = 0
  for user_id, movie_id, user_index, movie_index, rating in train_dataloader:
    optimizer.zero_grad()
    user_details  = user_tensor[user_index, 1:]
    movie_details = movies_tensor[movie_index, 1:]
    predictions = model(
      user_id,
      movie_id,
      user_details,
      movie_details
    )
    loss = criterion(predictions.squeeze(), rating.float())
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  if epoch % 5 == 0 and epoch != 0:
    model.eval()
    with torch.no_grad():
      user_details_test  = user_tensor[user_id_per_rating[test_data], 1:]
      movie_details_test = movies_tensor[movie_id_per_rating[test_data], 1:]
      predictions_test = model(
        rating_tensor[test_data, 0],
        rating_tensor[test_data, 1],
        user_details_test,
        movie_details_test
      )
      loss_test = criterion(predictions_test.squeeze(), rating_tensor[test_data, 2].float())
      loss_test = torch.sqrt(loss_test)
      train_rmse = torch.sqrt(torch.tensor(total_loss/len(train_dataloader)))
    print(f"Epoch {epoch+1}/{epochs}, Train RMSE: {train_rmse.item():.4f} | Test RMSE: {loss_test.item():.4f}")
  else:
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_dataloader):.4f}")
  scheduler.step()

Epoch 1/40, Loss: 2.7003
Epoch 2/40, Loss: 1.1753
Epoch 3/40, Loss: 1.0817
Epoch 4/40, Loss: 1.0187
Epoch 5/40, Loss: 0.9791
Epoch 6/40, Train RMSE: 0.9744 | Test RMSE: 0.9275
Epoch 7/40, Loss: 0.9276
Epoch 8/40, Loss: 0.9105
Epoch 9/40, Loss: 0.8942
Epoch 10/40, Loss: 0.8825
Epoch 11/40, Train RMSE: 0.9332 | Test RMSE: 0.9107
Epoch 12/40, Loss: 0.8599
Epoch 13/40, Loss: 0.8492
Epoch 14/40, Loss: 0.8390
Epoch 15/40, Loss: 0.8309
Epoch 16/40, Train RMSE: 0.9067 | Test RMSE: 0.9016
Epoch 17/40, Loss: 0.8143
Epoch 18/40, Loss: 0.8057
Epoch 19/40, Loss: 0.7995
Epoch 20/40, Loss: 0.7938
Epoch 21/40, Train RMSE: 0.8854 | Test RMSE: 0.8959
Epoch 22/40, Loss: 0.7804
Epoch 23/40, Loss: 0.7782
Epoch 24/40, Loss: 0.7757
Epoch 25/40, Loss: 0.7734
Epoch 26/40, Train RMSE: 0.8782 | Test RMSE: 0.8941
Epoch 27/40, Loss: 0.7685
Epoch 28/40, Loss: 0.7672
Epoch 29/40, Loss: 0.7652
Epoch 30/40, Loss: 0.7627
Epoch 31/40, Train RMSE: 0.8722 | Test RMSE: 0.8936
Epoch 32/40, Loss: 0.7592
Epoch 33/40, Loss: 0.

Spremanje modela

In [6]:
model_name = "model_v1.pth"
torch.save(model.state_dict(), model_name)
print(f"Model je uspješno spremljen na: {model_name}")

Model je uspješno spremljen na: model_v1.pth


Učitavanje istreniranog modela

In [3]:
model_url = "https://github.com/matej-glavas/NCF-movie-rec/raw/refs/heads/main/model_v1.pth"
model_path = "model_v1.pth"
if not os.path.exists(model_path):
  urllib.request.urlretrieve(model_url, model_path)
model.load_state_dict(torch.load(model_path, map_location="cpu"))
model.eval()
print(f"Model je uspješno učitan sa: {model_path}")

Model je uspješno učitan sa: model_v1.pth


Preporuka filmova na temelju sličnih korisnika i željenih žanrova (preporučuju se filmovi sa minimalno jednim od navedenih žanrova)

In [7]:
print("Gender? (M/F)" )
user_gender = input()
while(user_gender.capitalize() != "M" and user_gender.capitalize() != "F"):
  print("Gender? (M/F)" )
  user_gender = input()
if user_gender.capitalize() == "M":
  user_gender = 0
else:
  user_gender = 1
print("Age?\n  1: Under 18\n 18:  18-24\n 25:  25-34\n 35:  35-44\n 45:  45-49\n 50:  50-55\n 56:  56+")
user_age = input()
allowed_ages = ["1", "18", "25", "35", "45", "50", "56"]
while user_age not in allowed_ages:
  print("Age? ")
  user_age = input()
print("Occupation?\n  0:  other\n  1:  academic/educator\n  2:  artist\n  3:  clerical/admin\n  4:  college/grad student\n  5:  customer service\n  6:  doctor/health care\n  7:  executive/managerial\n  8:  farmer\n  9:  homemaker\n 10:  K-12 student\n 11:  lawyer\n 12:  programmer\n 13:  retired\n 14:  sales/marketing\n 15:  scientist\n 16:  self-employed\n 17:  technician/engineer\n 18:  tradesman/craftsman\n 19:  unemployed\n 20:  writer\n")
user_occupation = input()
allowed_occupations = []
for i in range(21):
  allowed_occupations.append(str(i))
while user_occupation not in allowed_occupations:
  print("Occupation? ")
  user_occupation = input()
details = np.array([int(user_gender), ages[int(user_age)], int(user_occupation)]).astype(np.float32)
current_user_details = torch.tensor(details).unsqueeze(0)

print("Genres?\n  Action: 0\n  Adventure: 1\n  Animation: 2\n  Children's: 3\n  Comedy: 4\n  Crime: 5\n  Documentary: 6\n  Drama: 7\n  Fantasy: 8\n  Film-Noir: 9\n  Horror: 10\n  Musical: 11\n  Mystery: 12\n  Romance: 13\n  Sci-Fi: 14\n  Thriller: 15\n  War: 16\n  Western: 17")
genre = input()
allowed_genres = []
user_genres = set()
for i in range(18):
  allowed_genres.append(str(i))
if genre in allowed_genres:
  user_genres.add(int(genre))
while(len(user_genres) == 0):
  print("Genre? ")
  genre = input()
  if genre in allowed_genres:
    user_genres.add(int(genre))
while genre != "X":
  print("Genre? Type X to stop.")
  genre = input()
  if genre in allowed_genres:
    user_genres.add(int(genre))

details = []
for i in range(18):
  if i in user_genres:
    details.append(1)
  else:
    details.append(0)
details = np.array(details).astype(np.float32)
current_movie_details = torch.tensor(details).unsqueeze(0)
recommended_movies = []
model.eval()
with torch.no_grad():
  for i in range(len(movies_tensor)):
    movie_details = movies_tensor[i, 1:].unsqueeze(0)
    movie_id = movies_tensor[i, 0].unsqueeze(0).long()
    if torch.logical_and(current_movie_details, movie_details).sum() >= 1:
      predictions = model(None, movie_id, current_user_details, movie_details)
      recommended_movies.append((movie_id.item(), predictions.item()))
recommended_movies.sort(key=lambda x: x[1], reverse=True)
print("Recommended movies:")
for i in range(5):
  for j in range(len(movies)):
    if str(recommended_movies[i][0]) == str(movies.values[j, 0]):
      print(movies.values[j][1])
      continue

Gender? (M/F)
m
Age?
  1: Under 18
 18:  18-24
 25:  25-34
 35:  35-44
 45:  45-49
 50:  50-55
 56:  56+
18
Occupation?
  0:  other
  1:  academic/educator
  2:  artist
  3:  clerical/admin
  4:  college/grad student
  5:  customer service
  6:  doctor/health care
  7:  executive/managerial
  8:  farmer
  9:  homemaker
 10:  K-12 student
 11:  lawyer
 12:  programmer
 13:  retired
 14:  sales/marketing
 15:  scientist
 16:  self-employed
 17:  technician/engineer
 18:  tradesman/craftsman
 19:  unemployed
 20:  writer

4
Genres?
  Action: 0
  Adventure: 1
  Animation: 2
  Children's: 3
  Comedy: 4
  Crime: 5
  Documentary: 6
  Drama: 7
  Fantasy: 8
  Film-Noir: 9
  Horror: 10
  Musical: 11
  Mystery: 12
  Romance: 13
  Sci-Fi: 14
  Thriller: 15
  War: 16
  Western: 17
15
Genre? Type X to stop.
X
Recommended movies:
Usual Suspects, The (1995)
Seven (Se7en) (1995)
Matrix, The (1999)
Green Mile, The (1999)
Sixth Sense, The (1999)


Usporedba istreniranog modela s klasičnim suradničkim filtriranjem

In [8]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

reader = Reader(rating_scale = (1,5))
data = Dataset.load_from_df(ratings[["user_id", "movie_id", "rating"]], reader)
cf_train_data, cf_test_data = train_test_split(data, test_size = 0.2)
svd = SVD()
svd.fit(cf_train_data)
predictions = svd.test(cf_test_data)
print("SVD ", end="")
accuracy.rmse(predictions)
print("SVD ", end="")
accuracy.mae(predictions)

print("---------------")

model.eval()
with torch.no_grad():
    user_details_test  = user_tensor[user_id_per_rating[test_data], 1:]
    movie_details_test = movies_tensor[movie_id_per_rating[test_data], 1:]
    predictions_test = model(
      rating_tensor[test_data, 0],
      rating_tensor[test_data, 1],
      user_details_test,
      movie_details_test
    ).squeeze()
    real_ratings = rating_tensor[test_data, 2]

    rmse_ncf = torch.sqrt(nn.MSELoss()(predictions_test, real_ratings)).item()
    mae_ncf  = nn.L1Loss()(predictions_test, real_ratings).item()

print(f"NCF RMSE: {rmse_ncf:.4f}")
print(f"NCF MAE:  {mae_ncf:.4f}")

SVD RMSE: 0.8744
SVD MAE:  0.6871
---------------
NCF RMSE: 0.8475
NCF MAE:  0.6717
